In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense , Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping
import os

In [ ]:
from google.colab import drive

path_dir = r"/content/drive/MyDrive/X-Ray_Image_Classification/chest_xray"

In [ ]:
!mkdir -p /content/chest_xray
!cp -r "/content/drive/MyDrive/X-Ray_Image_Classification/chest_xray/train" /content/chest_xray/
!cp -r "/content/drive/MyDrive/X-Ray_Image_Classification/chest_xray/test" /content/chest_xray/
!cp -r "/content/drive/MyDrive/X-Ray_Image_Classification/chest_xray/pred" /content/chest_xray/

In [ ]:
img_size = 100
base_model = VGG16(include_top=False , weights='imagenet' , input_shape=(img_size,img_size,3))

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
for layer in base_model.layers :
  layer.trainable = False

In [ ]:
x = Flatten()(base_model.output)
x = Dense(256)(x)
x = Dense(128 , activation = 'relu')(x)
x = Dense(64 , activation = 'relu')(x)
predictions = Dense(1 , activation = 'sigmoid')(x)

In [ ]:
model = Model(inputs=base_model.input , outputs=predictions)

In [ ]:
model.compile(optimizer = 'adam' , loss = 'binary_crossentropy' , metrics = ['accuracy'])

In [ ]:
train_data_gen = ImageDataGenerator(rescale=1/255 , rotation_range=10, zoom_range=0.4 , width_shift_range=0.1 , height_shift_range=0.1)
val_data_gen = ImageDataGenerator(rescale=1/255)
test_data_gen = ImageDataGenerator(rescale=1/255)

In [ ]:
train_dir = "/content/chest_xray/train"
val_dir = "/content/chest_xray/pred"
test_dir = "/content/chest_xray/test"

In [ ]:
aug_train_data = train_data_gen.flow_from_directory(train_dir , batch_size = 32 , target_size = (img_size,img_size) , class_mode="binary")
aug_val_data = val_data_gen.flow_from_directory(val_dir , batch_size = 32 , target_size = (img_size,img_size) , class_mode = "binary")


Found 7315 images belonging to 2 classes.
Found 9 images belonging to 2 classes.


In [ ]:
earlystop = EarlyStopping(monitor="val_loss" , patience=3 , restore_best_weights=True)

**When you set restore_best_weights=True, the model will restore the weights from the exact epoch that achieved the absolute lowest val_loss during the entire training run.**

In [ ]:
model.fit(aug_train_data , epochs = 10 , callbacks=[earlystop] , validation_data = aug_val_data)
#When you set restore_best_weights=True, the model will restore the weights from the exact epoch that achieved the absolute lowest val_loss during the entire training run.

Epoch 1/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 129s 529ms/step - accuracy: 0.8670 - loss: 0.3173 - val_accuracy: 1.0000 - val_loss: 0.1327
Epoch 2/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 111s 485ms/step - accuracy: 0.9087 - loss: 0.2374 - val_accuracy: 1.0000 - val_loss: 0.0254
Epoch 3/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 107s 469ms/step - accuracy: 0.9075 - loss: 0.2340 - val_accuracy: 1.0000 - val_loss: 0.0815
Epoch 4/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 109s 476ms/step - accuracy: 0.9161 - loss: 0.2226 - val_accuracy: 1.0000 - val_loss: 0.0828
Epoch 5/10
229/229 ━━━━━━━━━━━━━━━━━━━━ 107s 467ms/step - accuracy: 0.9170 - loss: 0.2081 - val_accuracy: 1.0000 - val_loss: 0.0682


In [ ]:
test_data = test_data_gen.flow_from_directory(test_dir , batch_size = 32 , target_size=(img_size,img_size) , class_mode = 'binary')

Found 620 images belonging to 2 classes.


In [ ]:
evaluation = model.evaluate(test_data)

20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 359ms/step - accuracy: 0.9097 - loss: 0.2684


In [ ]:
accuracy = evaluation[1]
print(accuracy)

0.9096774458885193


In [ ]:
path = r"/content/drive/MyDrive/X-Ray_Image_Classification/chest_xray/vgg16.h5"
model.save(path)